# Compare Base vs Fine-tuned Outputs (Google Colab)

Load `base_outputs.jsonl` and `finetuned_outputs.jsonl`, create a comparison table (10–15 questions), and optionally compute decision accuracy.

## Setup: Google Drive Structure

### Required files in `outputs/`

```
My Drive/
└── genai_hw3/
    └── outputs/
        ├── base_outputs.jsonl      ← From base_inference
        └── finetuned_outputs.jsonl ← From finetuned_inference_colab.ipynb
```

Both files must exist. Run `base_inference` and `finetuned_inference_colab` first to generate them.

## 1. Mount Google Drive & Load Data

In [ ]:
import json
import re
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# ========== Modify if using a different folder name ==========
PROJECT_FOLDER = "genai_hw3"

project_root = Path("/content/drive/MyDrive") / PROJECT_FOLDER
output_dir = project_root / "outputs"
base_path = output_dir / "base_outputs.jsonl"
finetuned_path = output_dir / "finetuned_outputs.jsonl"

def load_jsonl(path: Path) -> list[dict]:
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

base_results = load_jsonl(base_path) if base_path.exists() else []
ft_results = load_jsonl(finetuned_path) if finetuned_path.exists() else []

print(f"Base: {len(base_results)} records")
print(f"Fine-tuned: {len(ft_results)} records")

if not base_path.exists():
    raise FileNotFoundError(f"base_outputs.jsonl not found at {base_path}")
if not finetuned_path.exists():
    raise FileNotFoundError(f"finetuned_outputs.jsonl not found at {finetuned_path}")

## 2. Extract Decision from JSON Output

In [ ]:
def extract_decision(text: str) -> str | None:
    """Extract the decision field (over/under/avoid) from JSON string."""
    if not text or not isinstance(text, str):
        return None
    try:
        obj = json.loads(text)
        return obj.get("decision", "").lower().strip()
    except json.JSONDecodeError:
        pass
    m = re.search(r'"decision"\s*:\s*"([^"]+)"', text, re.IGNORECASE)
    return m.group(1).lower().strip() if m else None

# Test
sample = '{"decision": "over", "confidence": 0.7, ...}'
print(extract_decision(sample))

## 3. Compute Decision Accuracy

In [ ]:
def compute_accuracy(base_list: list[dict], ft_list: list[dict]) -> tuple[float, float]:
    """
    Compare model_output decision with ground_truth decision.
    Returns (base_acc, ft_acc).
    """
    n = min(len(base_list), len(ft_list))
    if n == 0:
        return 0.0, 0.0

    base_correct = ft_correct = total = 0
    for i in range(n):
        gt = extract_decision(base_list[i].get("ground_truth", ""))
        if not gt:
            continue
        total += 1
        b_pred = extract_decision(base_list[i].get("model_output", ""))
        f_pred = extract_decision(ft_list[i].get("model_output", "")) if i < len(ft_list) else None
        if b_pred == gt:
            base_correct += 1
        if f_pred == gt:
            ft_correct += 1

    base_acc = base_correct / total if total else 0
    ft_acc = ft_correct / total if total else 0
    return base_acc, ft_acc

base_acc, ft_acc = compute_accuracy(base_results, ft_results)
print(f"Base model decision accuracy: {base_acc:.2%}")
print(f"Fine-tuned model decision accuracy: {ft_acc:.2%}")

## 4. Create Comparison Table (10–15 Questions)

In [ ]:
NUM_SAMPLES = 15  # At least 10–15 questions for assignment

n = min(NUM_SAMPLES, len(base_results), len(ft_results))
comparisons = []

for i in range(n):
    b = base_results[i]
    f = ft_results[i] if i < len(ft_results) else {}
    gt_decision = extract_decision(b.get("ground_truth", ""))
    base_decision = extract_decision(b.get("model_output", ""))
    ft_decision = extract_decision(f.get("model_output", ""))
    comparisons.append({
        "idx": i + 1,
        "question": b.get("question", "")[:80] + "..." if len(b.get("question", "")) > 80 else b.get("question", ""),
        "ground_truth": gt_decision or "-",
        "base_decision": base_decision or "-",
        "ft_decision": ft_decision or "-",
        "base_match": "✓" if base_decision == gt_decision else "✗",
        "ft_match": "✓" if ft_decision == gt_decision else "✗",
    })

for c in comparisons:
    print(f"{c['idx']}. {c['question']}")
    print(f"   GT: {c['ground_truth']} | Base: {c['base_decision']} ({c['base_match']}) | FT: {c['ft_decision']} ({c['ft_match']})")
    print()

## 5. Export Sample Data for sample_outputs.md

In [ ]:
sample_data = []
for i in range(min(5, len(base_results), len(ft_results))):
    b = base_results[i]
    f = ft_results[i] if i < len(ft_results) else {}
    sample_data.append({
        "question": b.get("question", ""),
        "ground_truth": b.get("ground_truth", ""),
        "base_output": b.get("model_output", ""),
        "finetuned_output": f.get("model_output", ""),
    })

sample_path = project_root / "sample_outputs_data.json"
with open(sample_path, "w", encoding="utf-8") as f:
    json.dump(sample_data, f, ensure_ascii=False, indent=2)

print(f"Sample data exported to: {sample_path}")

## 6. Generate sample_outputs.md

In [ ]:
def generate_sample_md(sample_data: list[dict], out_path: Path) -> None:
    lines = [
        "# Sample Outputs: Base vs Fine-tuned Model",
        "",
        "This document compares outputs from the Base Model (SmolLM2-1.7B-Instruct) and the Fine-tuned Model (LoRA).",
        "",
        "## Decision Accuracy Summary",
        "",
        f"- **Base model**: {base_acc:.2%}",
        f"- **Fine-tuned model**: {ft_acc:.2%}",
        "",
        "---",
        "",
    ]
    for i, d in enumerate(sample_data, 1):
        lines.extend([
            f"## Example {i}",
            "",
            "### Question",
            "",
            f"{d['question']}",
            "",
            "### Ground Truth",
            "",
            "```json",
            d["ground_truth"][:1500] + ("..." if len(d["ground_truth"]) > 1500 else ""),
            "```",
            "",
            "### Base Model Output",
            "",
            "```json",
            d["base_output"][:1500] + ("..." if len(d["base_output"]) > 1500 else ""),
            "```",
            "",
            "### Fine-tuned Model Output",
            "",
            "```json",
            d["finetuned_output"][:1500] + ("..." if len(d["finetuned_output"]) > 1500 else ""),
            "```",
            "",
            "---",
            "",
        ])
    lines.extend([
        "## Discussion",
        "",
        "- **Format**: The fine-tuned model should output JSON that more consistently follows the Tree of Thought structure.",
        "- **Reasoning**: After fine-tuning, the model should better interpret n_games, p_over/p_under, starter vs bench, etc.",
        "- **Decision**: Observe whether decision accuracy improves and whether confidence scores are more reasonable.",
        "",
    ])
    out_path.write_text("\n".join(lines), encoding="utf-8")

md_path = project_root / "sample_outputs.md"
generate_sample_md(sample_data, md_path)
print(f"Generated: {md_path}")